# Lab 3 — Model Registry y Serving con FastAPI

**Taller: MLOps en la práctica — del notebook a producción** · UNI
**Duración:** Parte A (Registry, ~40 min, sábado) + Parte B (Serving, ~65 min, domingo)

## 🎯 Objetivos
1. Convertir "el run ganador" en un **modelo registrado y versionado** con ciclo de vida (Registry).
2. Promover modelos con **alias** (`champion` / `challenger`) — el mecanismo de despliegue sin tocar código.
3. Exponer el modelo como **API REST** con FastAPI + validación de entrada con Pydantic.
4. Probar la API como lo haría el sistema del call center de AndesTel.

> ⚠️ Este lab continúa el Lab 2: necesita `mlflow.db` y `churn_telco_peru.csv` en el mismo directorio.
> Si vienes de una sesión nueva de Colab, re-ejecuta primero el Lab 2 (2 min) o pide al instructor el `mlflow.db` de respaldo.

In [1]:
%pip install -q pandas pyarrow fastapi uvicorn requests

Note: you may need to restart the kernel to use updated packages.


# Parte A — Model Registry (sábado)

## 1. ¿Qué problema resuelve el Registry?

El tracking responde *"¿qué experimentos corrimos?"*. El **Registry** responde otra pregunta:
*"¿Cuál es EL modelo oficial de churn, qué versión está en producción, y quién aprobó el cambio?"*

Es el puente entre el equipo de ciencia de datos y el de operaciones:

```
Experimentos (cientos de runs)
        │  registrar el campeón
        ▼
Modelo registrado: "churn-andestel"   ← nombre único en la organización
        ├── versión 1  (alias: ninguno — quedó obsoleta)
        ├── versión 2  (alias: champion  → la que sirve producción)
        └── versión 3  (alias: challenger → candidata en evaluación)
```

In [1]:
import mlflow
from mlflow import MlflowClient

mlflow.set_tracking_uri("sqlite:///mlflow.db")
client = MlflowClient()

# Recuperamos el run campeón del Lab 2
with open("run_id_campeon.txt") as f:
    RUN_ID_CAMPEON = f.read().strip()
print("Run campeón:", RUN_ID_CAMPEON)

c:\Users\Theki\miniconda3\envs\ml_pro\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Run campeón: 10284b4ab8724bd7838ade7000b95df0


In [2]:
# === Registrar el modelo del run campeón bajo un nombre oficial ===
NOMBRE_MODELO = "churn-andestel"

version = mlflow.register_model(
    model_uri=f"runs:/{RUN_ID_CAMPEON}/modelo",
    name=NOMBRE_MODELO,
)
print(f"Registrado '{NOMBRE_MODELO}' como versión {version.version}")

Successfully registered model 'churn-andestel'.
2026/08/29 18:40:24 WARNING mlflow.tracking._model_registry.fluent: Run with id 10284b4ab8724bd7838ade7000b95df0 has no artifacts at artifact path 'modelo', registering model based on models:/m-1c83e6e6d58c4744aee3e499f6719ab7 instead


Registrado 'churn-andestel' como versión 1


Created version '1' of model 'churn-andestel'.


In [3]:
# === Promoverlo: asignarle el alias "champion" ===
client.set_registered_model_alias(NOMBRE_MODELO, "champion", version.version)

# Documentar la decisión (esto es lo que un auditor o tu yo-futuro agradecerá)
client.update_model_version(
    name=NOMBRE_MODELO, version=version.version,
    description=("Seleccionado en barrido sesion1 por criterio: AUC top ±0.01, mayor recall. "
                 "Dataset churn_telco_peru_2025. Aprobado en taller UNI."),
)
print(f"✅ '{NOMBRE_MODELO}' v{version.version} ahora es champion")

✅ 'churn-andestel' v1 ahora es champion


In [4]:
# === El contrato clave: cualquier sistema carga el modelo POR ALIAS, sin saber la versión ===
modelo_prod = mlflow.sklearn.load_model(f"models:/{NOMBRE_MODELO}@champion")

import pandas as pd
cliente = pd.DataFrame([{
    "edad": 45, "departamento": "Lima", "plan": "Postpago Plus", "tipo_contrato": "Anual",
    "meses_antiguedad": 30, "cargo_mensual_soles": 72.5, "gb_datos_mes": 25.0,
    "minutos_llamadas_mes": 350, "lineas_adicionales": 1, "tickets_soporte_6m": 0,
    "caidas_servicio_mes": 1, "dias_ultimo_pago_vencido": 0, "factura_electronica": 1,
}])
print(f"P(churn) según el champion: {modelo_prod.predict_proba(cliente)[0, 1]:.1%}")

P(churn) según el champion: 16.8%


### 💡 Por qué esto es oro en tu trabajo

El código de producción dice `@champion` y **nunca cambia**. Desplegar un modelo nuevo = registrar versión + mover el alias. Rollback = mover el alias de vuelta. Sin re-deploy, sin editar código, con historial completo de quién movió qué y cuándo.

### 🎯 Reto A (10 min)
Entrena una variante rápida (cambia un hiperparámetro), regístrala como nueva versión del mismo nombre y asígnale el alias `challenger`. Verifica con `client.get_model_version_by_alias(NOMBRE_MODELO, "challenger")`. Acabas de montar la base de un despliegue *shadow/champion-challenger*.

# Parte B — Serving con FastAPI (domingo)

## 2. Del modelo cargable al modelo consultable

El call center de AndesTel usa un CRM hecho en otro lenguaje por otro equipo. No van a importar tu pipeline de Python: necesitan un **servicio HTTP** — le mandan un JSON con los datos del cliente y reciben la probabilidad de churn.

**FastAPI** es el estándar actual en Python para esto: rápido, con validación automática (Pydantic) y documentación interactiva gratis (`/docs`).

La celda siguiente escribe `app.py` — el archivo del servicio, tal como viviría en un repo:

In [7]:
%%writefile app.py
"""Servicio de predicción de churn — AndesTel. Taller MLOps UNI."""
from contextlib import asynccontextmanager
from enum import Enum
from typing import Optional

import mlflow
import pandas as pd
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel, Field

NOMBRE_MODELO = "churn-andestel"
UMBRAL_ALERTA = 0.35  # decidido con negocio (Lab 1): priorizamos recall

estado = {"modelo": None, "version": None}

@asynccontextmanager
async def lifespan(app: FastAPI):
    # Al arrancar: cargar el champion desde el Registry (una sola vez, no por request)
    mlflow.set_tracking_uri("sqlite:///mlflow.db")
    client = mlflow.MlflowClient()
    mv = client.get_model_version_by_alias(NOMBRE_MODELO, "champion")
    estado["modelo"] = mlflow.sklearn.load_model(f"models:/{NOMBRE_MODELO}@champion")
    estado["version"] = mv.version
    yield
    estado["modelo"] = None

app = FastAPI(title="API Churn AndesTel", version="1.0", lifespan=lifespan)

# --- Contrato de entrada: Pydantic valida tipos y rangos ANTES de llegar al modelo ---
class Plan(str, Enum):
    prepago = "Prepago"
    basico = "Postpago Básico"
    plus = "Postpago Plus"
    premium = "Postpago Premium"

class Contrato(str, Enum):
    mensual = "Mensual"
    anual = "Anual"
    m18 = "18 meses"

class Cliente(BaseModel):
    edad: Optional[float] = Field(None, ge=18, le=100)
    departamento: str
    plan: Plan
    tipo_contrato: Contrato
    meses_antiguedad: int = Field(..., ge=1, le=200)
    cargo_mensual_soles: float = Field(..., ge=0)
    gb_datos_mes: Optional[float] = Field(None, ge=0)
    minutos_llamadas_mes: int = Field(..., ge=0)
    lineas_adicionales: int = Field(0, ge=0, le=10)
    tickets_soporte_6m: int = Field(0, ge=0)
    caidas_servicio_mes: int = Field(0, ge=0)
    dias_ultimo_pago_vencido: int = Field(0, ge=0)
    factura_electronica: int = Field(..., ge=0, le=1)

class Prediccion(BaseModel):
    probabilidad_churn: float
    riesgo: str
    accion_sugerida: str
    version_modelo: str

@app.get("/health")
def health():
    return {"status": "ok", "modelo": NOMBRE_MODELO, "version": estado["version"]}

@app.post("/predict", response_model=Prediccion)
def predict(cliente: Cliente):
    if estado["modelo"] is None:
        raise HTTPException(503, "Modelo no cargado")
    df = pd.DataFrame([cliente.model_dump()])
    # Pydantic entrega Enums; el pipeline espera strings
    df["plan"] = df["plan"].map(lambda x: x.value if hasattr(x, "value") else x)
    df["tipo_contrato"] = df["tipo_contrato"].map(lambda x: x.value if hasattr(x, "value") else x)
    proba = float(estado["modelo"].predict_proba(df)[0, 1])
    riesgo = "alto" if proba >= UMBRAL_ALERTA else ("medio" if proba >= 0.20 else "bajo")
    accion = {
        "alto": "Derivar a retención con oferta personalizada",
        "medio": "Incluir en campaña de fidelización del mes",
        "bajo": "Sin acción",
    }[riesgo]
    return Prediccion(probabilidad_churn=round(proba, 4), riesgo=riesgo,
                      accion_sugerida=accion, version_modelo=str(estado["version"]))

Writing app.py


### Anatomía del servicio (léelo antes de ejecutarlo)

- **`lifespan`**: el modelo se carga **una vez al arrancar** desde el Registry (`@champion`), no en cada request. Error clásico: cargar el modelo dentro del endpoint → latencia de segundos por llamada.
- **`Cliente(BaseModel)`**: el *contrato de entrada*. Si el CRM manda `edad: "cuarenta"` o un plan inexistente, FastAPI responde **422 con el detalle del error** sin que el modelo se entere. La validación de entrada es tu primera línea de defensa en producción.
- **`Prediccion`**: el contrato de salida incluye `version_modelo` → cada respuesta es auditable ("¿qué modelo te dijo eso?").
- **El umbral y la acción** viven en el servicio como reglas de negocio explícitas.

In [8]:
# === Levantar el servicio DENTRO del notebook (truco para talleres; en la vida real: terminal/Docker) ===
import threading, time
import uvicorn

def _run():
    uvicorn.run("app:app", host="127.0.0.1", port=8000, log_level="warning")

hilo = threading.Thread(target=_run, daemon=True)
hilo.start()
time.sleep(8)  # dar tiempo a cargar el modelo
print("Servicio arriba (si la siguiente celda falla, espera 5 s y reintenta)")

Servicio arriba (si la siguiente celda falla, espera 5 s y reintenta)


In [9]:
import requests

r = requests.get("http://127.0.0.1:8000/health", timeout=10)
print(r.status_code, r.json())

200 {'status': 'ok', 'modelo': 'churn-andestel', 'version': 1}


In [11]:
%%time
# === Así consulta el CRM del call center ===
cliente_riesgoso = {
    "edad": 24, "departamento": "Piura", "plan": "Prepago", "tipo_contrato": "Mensual",
    "meses_antiguedad": 2, "cargo_mensual_soles": 35.0, "gb_datos_mes": 8.0,
    "minutos_llamadas_mes": 120, "lineas_adicionales": 0, "tickets_soporte_6m": 5,
    "caidas_servicio_mes": 4, "dias_ultimo_pago_vencido": 15, "factura_electronica": 0,
}
r = requests.post("http://127.0.0.1:8000/predict", json=cliente_riesgoso, timeout=10)
print(r.status_code)
r.json()

200
CPU times: total: 31.2 ms
Wall time: 24.5 ms


{'probabilidad_churn': 0.9876,
 'riesgo': 'alto',
 'accion_sugerida': 'Derivar a retención con oferta personalizada',
 'version_modelo': '1'}

In [12]:
# === La validación en acción: datos malos NO llegan al modelo ===
cliente_invalido = dict(cliente_riesgoso, edad=15, plan="Plan Pirata")
r = requests.post("http://127.0.0.1:8000/predict", json=cliente_invalido, timeout=10)
print("Status:", r.status_code, "(422 = entrada rechazada por el contrato)")
for e in r.json()["detail"]:
    print(" -", e["loc"], "->", e["msg"])

Status: 422 (422 = entrada rechazada por el contrato)
 - ['body', 'edad'] -> Input should be greater than or equal to 18
 - ['body', 'plan'] -> Input should be 'Prepago', 'Postpago Básico', 'Postpago Plus' or 'Postpago Premium'


### 📚 La documentación que no escribiste

FastAPI genera documentación interactiva automáticamente. **En local:** abre http://127.0.0.1:8000/docs — puedes probar la API desde el navegador. **En Colab:** ejecuta la celda siguiente.

In [ ]:
# Solo Colab: abrir /docs vía proxy de puertos
try:
    from google.colab import output  # noqa
    output.serve_kernel_port_as_window(8000, path="/docs")
except ImportError:
    print("No estás en Colab: abre http://127.0.0.1:8000/docs en tu navegador")

## 🎯 Retos (20 min)

**Reto 1 (todos):** agrega un endpoint `POST /predict_lote` que reciba una **lista** de clientes (`list[Cliente]`) y devuelva la lista de predicciones ordenada por probabilidad descendente (los primeros a llamar). Edita `app.py`, y reinicia el kernel para relanzar el servicio (o usa un puerto nuevo).

**Reto 2 (todos):** rompe el contrato de 3 formas distintas (tipo incorrecto, campo faltante, valor fuera de rango) y lee con cuidado los errores 422. Este es el lenguaje con el que hablarás con el equipo que integra tu API.

**Reto 3 (avanzado):** agrega al response un campo `factores_principales` con las 3 features de mayor importancia global del modelo (pista: `modelo.named_steps`, importancias del clasificador y nombres de columnas del preprocesador).

**Reto 4 (avanzado):** mide la latencia: manda 200 requests en bucle y calcula p50 y p95 con `numpy.percentile`. ¿Cumplirías un SLA de 100 ms?

## 📌 Lo que te llevas

- ✅ Registry: nombre oficial + versiones + alias = despliegue y rollback moviendo un puntero, con historial.
- ✅ El código de producción referencia `@champion`, nunca una versión fija.
- ✅ FastAPI + Pydantic: contrato de entrada/salida explícito, validación automática, docs gratis.
- ✅ El modelo se carga al arrancar el servicio; cada respuesta declara qué versión respondió.

**Problema pendiente:** "en mi máquina funciona" otra vez — ¿cómo llevo este servicio a un servidor real, idéntico? → **Lab 4: Docker + CI/CD** (con el repo plantilla).